In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

DATA    = Path.cwd().parent / 'data' / 'interim'
FIGURES = Path.cwd().parent / 'reports' / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)

BLUE, RED, ORANGE, GREEN, GRAY = '#2563EB','#DC2626','#F59E0B','#16A34A','#6B7280'

msg_onboarding  = pd.read_csv(DATA / 'msg_onboarding.csv')
conv_onboarding = pd.read_csv(DATA / 'conv_onboarding.csv')
features        = pd.read_csv(DATA / 'features.csv')

# tx_with_reg has days_since_reg; compute days_since_first_tx from first tx per client
tx_raw = pd.read_csv(DATA / 'tx_with_reg.csv')
first_tx_day = tx_raw.groupby('client_id')['days_since_reg'].min().rename('first_tx_reg_day')
tx_raw = tx_raw.join(first_tx_day, on='client_id')
tx_raw['days_since_first_tx'] = tx_raw['days_since_reg'] - tx_raw['first_tx_reg_day']

print(f'msg_onboarding:  {len(msg_onboarding):,}')
print(f'conv_onboarding: {len(conv_onboarding):,}')
print(f'tx_raw:          {len(tx_raw):,}')


In [ ]:
# Conversion rate by onboarding week
msg_onboarding['week'] = (msg_onboarding['days_since_first_tx'] // 7 + 1).clip(1,13).astype(int)
conv_with_week = conv_onboarding.merge(
    msg_onboarding[['message_id','week','segment_group']], on='message_id', how='left'
)
timing = pd.concat([
    msg_onboarding.groupby('week')['message_id'].count().rename('n_msg'),
    conv_with_week.groupby('week')['conversion_id'].count().rename('n_conv'),
], axis=1).fillna(0).reset_index()
timing['conv_rate'] = timing['n_conv'] / timing['n_msg'] * 100

print(f"{'Week':>5}  {'Msgs':>9}  {'Conv':>8}  {'Rate':>7}")
print('-' * 35)
for _, row in timing.iterrows():
    marker = '  <-- peak' if row['conv_rate'] == timing['conv_rate'].max() else ''
    print(f"{int(row['week']):>5}  {int(row['n_msg']):>9,}  {int(row['n_conv']):>8,}  {row['conv_rate']:>6.1f}%{marker}")


In [ ]:
# Hazard function: when do multi-visit churners make their last transaction?
fading_mask = (features['churned'] == 1) & (features['tx_count'] > 1)
fading_ids  = features.loc[fading_mask, 'client_id']

# Last active day within the 90-day window for each fading client
fading_tx = tx_raw[
    tx_raw['client_id'].isin(fading_ids) &
    (tx_raw['days_since_first_tx'] > 0) &
    (tx_raw['days_since_first_tx'] <= 90)
]
last_day = (
    fading_tx.groupby('client_id')['days_since_first_tx'].max()
    .reset_index().rename(columns={'days_since_first_tx':'last_active_day'})
)
last_day['day'] = last_day['last_active_day'].round().astype(int).clip(0, 90)

n_total = len(last_day)
hazard_rows = []
for t in range(0, 91):
    n_at_risk = (last_day['last_active_day'] >= t).sum()
    n_events  = (last_day['day'] == t).sum()
    h_t = n_events / n_at_risk if n_at_risk > 0 else 0
    hazard_rows.append({'day': t, 'n_at_risk': n_at_risk, 'n_events': n_events, 'hazard': h_t})
hazard_df = pd.DataFrame(hazard_rows)

acute   = hazard_df[hazard_df['day'] <= 7]['n_events'].sum()
chronic = hazard_df[hazard_df['day'] > 7]['n_events'].sum()
print(f'Multi-visit churners (fading): {n_total:,}')
print(f'Acute   phase (days  0-7): {acute:,} ({acute/n_total*100:.1f}%)')
print(f'Chronic phase (days 8-90): {chronic:,} ({chronic/n_total*100:.1f}%)')


In [ ]:
# Chronic fader survival: how many still active at day t?
chronic_df = hazard_df[hazard_df['day'] >= 8].copy()
chronic_start = chronic_df['n_at_risk'].iloc[0]
chronic_df['pct_alive'] = chronic_df['n_at_risk'] / chronic_start * 100

print(f'Chronic faders at day 8: {chronic_start:,} (100%)')
for day in [14, 21, 30, 45, 60, 75, 90]:
    row = chronic_df[chronic_df['day'] == day]
    if len(row):
        print(f'  Day {day:2d}: {int(row["n_at_risk"].values[0]):,} ({row["pct_alive"].values[0]:.1f}% alive)')


In [ ]:
# Hazard figure (thesis: hazard_function.png)
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(hazard_df['day'], hazard_df['hazard'], color=BLUE, lw=2.2, zorder=3)
ax.axvspan(0,  7,  alpha=0.15, color=RED,   zorder=1)
ax.axvspan(30, 45, alpha=0.18, color=GREEN, zorder=1)
ax.text(3.5, hazard_df['hazard'].max()*0.82, 'Acute Phase\n(Unreachable)',
        ha='center', va='top', fontsize=9, color='#991B1B',
        bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='#991B1B', alpha=0.7))
ax.set_xlabel('Day of Onboarding Window', fontsize=11)
ax.set_ylabel('Hazard Rate h(t)', fontsize=11)
ax.set_title('Hazard Function -- Fading Churners', fontsize=13, fontweight='bold')
ax.set_xlim(0, 90); ax.set_ylim(bottom=0)
ax.legend(handles=[
    mpatches.Patch(facecolor=RED,   alpha=0.4, label='Acute phase (days 0-7)'),
    mpatches.Patch(facecolor=GREEN, alpha=0.4, label='Optimal retention window (days 30-45)'),
], fontsize=9)
plt.tight_layout()
plt.savefig(FIGURES / 'hazard_function.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Kaplan-Meier: time to second visit by first-message timing group
second_visit = (
    tx_raw[tx_raw['days_since_first_tx'] > 0]
    .groupby('client_id')['days_since_first_tx'].min()
    .reset_index().rename(columns={'days_since_first_tx':'day_second_visit'})
)
clients = features[['client_id','churned','tx_count']].copy()
km_data = clients.merge(second_visit, on='client_id', how='left')
km_data['event']    = (km_data['day_second_visit'].notna() & (km_data['day_second_visit'] <= 90)).astype(int)
km_data['duration'] = km_data['day_second_visit'].clip(upper=90).fillna(90)

first_msg = (
    msg_onboarding.sort_values('days_since_first_tx')
    .groupby('client_id').first().reset_index()
    [['client_id','days_since_first_tx','segment_group']]
    .rename(columns={'days_since_first_tx':'first_msg_day','segment_group':'first_msg_seg'})
)
km_data = km_data.merge(first_msg, on='client_id', how='left')

def timing_group(day):
    if pd.isna(day): return 'No messages'
    if day <= 7:     return 'Days 0-7'
    if day <= 14:    return 'Days 7-14'
    if day <= 30:    return 'Days 14-30'
    return 'Days 30+'
km_data['timing_group'] = km_data['first_msg_day'].apply(timing_group)

print(km_data.groupby('timing_group')[['event','client_id']]
      .agg(n=('client_id','count'), second_visit_rate=('event','mean')).round(3))


In [ ]:
# KM estimator (no external libraries)
def kaplan_meier(df, dur='duration', ev='event', max_t=90):
    timeline = sorted(df[df[ev]==1][dur].unique())
    rows = [{'t': 0, 'S': 1.0}]
    S = 1.0
    for t in timeline:
        if t > max_t: break
        n_risk   = (df[dur] >= t).sum()
        n_events = ((df[dur] == t) & (df[ev] == 1)).sum()
        if n_risk > 0: S *= (1 - n_events / n_risk)
        rows.append({'t': t, 'S': S})
    return pd.DataFrame(rows)

groups  = ['No messages','Days 0-7','Days 7-14','Days 14-30','Days 30+']
colors  = [GRAY, RED, ORANGE, BLUE, GREEN]
km_res  = {g: kaplan_meier(km_data[km_data['timing_group']==g]) for g in groups}

# Print S(90) = churn-free survival at day 90
print(f"{'Group':20s}  {'n':>8}  {'Return rate':>12}  {'S(90)':>8}")
for g in groups:
    n = len(km_data[km_data['timing_group']==g])
    s90 = km_res[g][km_res[g]['t'] <= 90]['S'].iloc[-1]
    ev_rate = km_data[km_data['timing_group']==g]['event'].mean()
    print(f'{g:20s}  {n:>8,}  {ev_rate:>12.3f}  {s90:>8.3f}')


In [ ]:
# KM figure (thesis: kaplan_meier_timing.png)
fig, ax = plt.subplots(figsize=(9, 6))
for g, color in zip(groups, colors):
    km  = km_res[g]
    n   = len(km_data[km_data['timing_group']==g])
    lw  = 2.5 if g == 'Days 0-7' else 1.8
    ax.step(km['t'], km['S'], where='post', color=color, lw=lw, label=f'{g}  (n={n:,})')
ax.set_xlabel('Day of Onboarding Window', fontsize=11)
ax.set_ylabel('Probability of Not Returning S(t)', fontsize=11)
ax.set_title('Kaplan-Meier Survival Curves by First Message Timing', fontsize=11)
ax.set_xlim(0, 90); ax.set_ylim(0, 1.0)
ax.legend(fontsize=9, loc='upper right')
plt.tight_layout()
plt.savefig(FIGURES / 'kaplan_meier_timing.png', dpi=150, bbox_inches='tight')
plt.show()
